# Consumer Spending Intelligence
## End-to-End Financial Analytics Pipeline | Credit Card Transactions

**Author:** Sujal Deb  
**Dataset:** Credit Card Transactions | 1.296M records | Jan 2019 – Jun 2020  
**Tools:** Python, Pandas, Matplotlib, Plotly, Statsmodels, Scikit-learn, Streamlit

---

## Business Context

A financial services company processes credit card transactions across 983 customers,
693 merchants, and 14 spending categories spanning 18 months across the United States.
Despite having a large volume of transactional data, the business has no unified system to:

- Understand how consumer spending behaviour evolves over time
- Identify which customer segments, categories, and merchants drive revenue
- Detect statistically significant shifts in spending patterns
- Flag anomalous transactions that deviate from established behaviour
- Quantify geographic and demographic drivers of spend

---

## Problem Statement

> **How do consumer spending patterns evolve over time across categories, demographics,
> and geographies — and where do statistically significant shifts, anomalies, and
> growth opportunities exist within this transaction portfolio?**

---

## Analytical Objectives

| # | Objective | Notebook |
|---|-----------|----------|
| 1 | Validate and profile the raw transaction dataset | 01 — Ingestion |
| 2 | Understand distributions, category mix, and demographic composition | 02 — EDA |
| 3 | Decompose spending trends, seasonality, and test temporal hypotheses | 03 — Trends |
| 4 | Segment customers by behaviour and validate demographic spend differences | 04 — Customer Intelligence |
| 5 | Map geographic revenue concentration and merchant performance | 05 — Geo & Merchant |
| 6 | Detect anomalous transactions using unsupervised learning | 06 — Anomaly Detection |
| 7 | Detect structural changepoints in consumer spending time series | 07 — Behaviour Shift |

---

## Dataset Schema (post-cleaning)

| Column | Type | Description |
|--------|------|-------------|
| trans_date_trans_time | datetime | Transaction timestamp |
| cc_num | string | Customer identifier |
| merchant | category | Merchant name |
| category | category | Spending category |
| amt | float | Transaction amount in USD |
| gender | category | Customer gender |
| city, state, zip | string/category | Customer location |
| lat, long | float | Customer coordinates |
| city_pop | int | Population of customer city |
| job | category | Customer occupation |
| dob | datetime | Customer date of birth |
| trans_num | string | Unique transaction identifier |
| merch_lat, merch_long | float | Merchant coordinates |
| is_fraud | int8 | Fraud label — 1 = fraudulent |
| merch_zipcode | string | Merchant zip code |

---

## Project Structure
## Project Structure

    Consumer-Spending-Intelligence/
    ├── data/
    │   ├── raw/                        <- original CSV from Kaggle
    │   └── processed/                  <- cleaned Parquet files
    ├── notebooks/
    │   ├── 01_data_ingestion_and_validation.ipynb
    │   ├── 02_exploratory_data_analysis.ipynb
    │   ├── 03_spending_trends.ipynb
    │   ├── 04_customer_intelligence.ipynb
    │   ├── 05_geo_merchant_analysis.ipynb
    │   ├── 06_anomaly_detection.ipynb
    │   └── 07_behaviour_shift_detection.ipynb
    ├── app/
    │   └── streamlit_app.py
    ├── requirements.txt
    └── README.md

---

*This notebook covers Objective 1 — data ingestion, validation, and Parquet serialisation.*


# Notebook 01 — Data Ingestion & Validation

## Consumer Spending Intelligence | Credit Card Transactions Dataset
**Dataset:** 1.85M credit card transactions | Kaggle (priyamchoksi)  
**Notebook goal:** Load the raw CSV, understand its structure, validate data quality, and save a clean Parquet file for all downstream notebooks.

---

### What this notebook covers
1. Load raw CSV and confirm shape and schema
2. Parse and validate all data types
3. Audit nulls, duplicates, and anomalous values
4. Profile the date range and transaction volume timeline
5. Save validated data as Parquet for fast downstream loading

---

> **Why Parquet?**  
> The raw CSV is ~500MB. Parquet compresses it to ~60MB and loads 10x faster in every subsequent notebook. All notebooks after this one load from Parquet, never from the raw CSV.

In [2]:

# IMPORTS & CONFIGURATION
# All imports for this notebook declared upfront for clarity and reproducibility


import pandas as pd
import numpy as np
import os
import warnings

warnings.filterwarnings("ignore")

#  Path configuration 
# Using relative paths so the project works on any machine
RAW_DATA_PATH   = "../data/raw/credit_card_transactions.csv"
PROCESSED_DIR   = "../data/processed/"
PARQUET_PATH    = os.path.join(PROCESSED_DIR, "transactions_clean.parquet")

# Create processed directory if it doesn't already exist
os.makedirs(PROCESSED_DIR, exist_ok=True)

#  Display configuration 
pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", "{:,.2f}".format)

print(" Libraries imported successfully")
print(f" Raw data path   : {RAW_DATA_PATH}")
print(f" Processed dir   : {PROCESSED_DIR}")
print(f" Output path     : {PARQUET_PATH}")

 Libraries imported successfully
 Raw data path   : ../data/raw/credit_card_transactions.csv
 Processed dir   : ../data/processed/
 Output path     : ../data/processed/transactions_clean.parquet


## Step 1 — Load Raw Data

Load the CSV and immediately inspect shape, column names, and data types.
We do not make any modifications here — this is a read-only inspection step.

In [3]:
# Load raw CSV
df = pd.read_csv(RAW_DATA_PATH)

# Basic shape audit
print(f"Rows         : {df.shape[0]:,}")
print(f"Columns      : {df.shape[1]}")
print(f"Memory usage : {df.memory_usage(deep=True).sum() / 1e6:.1f} MB")

# Column names and dtypes
print("\nColumn overview:")
print(df.dtypes.to_string())

Rows         : 1,296,675
Columns      : 24
Memory usage : 1095.6 MB

Column overview:
Unnamed: 0                 int64
trans_date_trans_time     object
cc_num                     int64
merchant                  object
category                  object
amt                      float64
first                     object
last                      object
gender                    object
street                    object
city                      object
state                     object
zip                        int64
lat                      float64
long                     float64
city_pop                   int64
job                       object
dob                       object
trans_num                 object
unix_time                  int64
merch_lat                float64
merch_long               float64
is_fraud                   int64
merch_zipcode            float64


## Step 2 — Data Type Casting & Memory Optimisation

The raw CSV loads everything as default dtypes which wastes significant memory.
We cast each column to its correct logical type.

Changes made:
- Drop `Unnamed: 0` — CSV artifact, carries no information
- Parse `trans_date_trans_time` and `dob` as datetime
- Cast `cc_num`, `zip`, `merch_zipcode` to string — these are identifiers, not numbers
- Cast `gender`, `category`, `state`, `job` to category dtype — reduces memory for low-cardinality columns
- Cast `is_fraud` to int8 — binary flag needs only 1 byte not 8

In [4]:
# drop the csv row index artifact
df.drop(columns=["Unnamed: 0"], inplace=True)

# parse datetime columns
df["trans_date_trans_time"] = pd.to_datetime(df["trans_date_trans_time"])
df["dob"] = pd.to_datetime(df["dob"])

# identifier columns — cast to string, not numeric
df["cc_num"]        = df["cc_num"].astype(str)
df["zip"]           = df["zip"].astype(str)
df["merch_zipcode"] = df["merch_zipcode"].astype(str)

# low-cardinality columns — category dtype cuts memory significantly
for col in ["gender", "category", "state", "job"]:
    df[col] = df[col].astype("category")

# binary flag — int8 is sufficient
df["is_fraud"] = df["is_fraud"].astype("int8")

# confirm results
print(f"Memory after optimisation : {df.memory_usage(deep=True).sum() / 1e6:.1f} MB")
print(f"Shape                     : {df.shape}")
print("\nUpdated dtypes:")
print(df.dtypes.to_string())

Memory after optimisation : 835.0 MB
Shape                     : (1296675, 23)

Updated dtypes:
trans_date_trans_time    datetime64[ns]
cc_num                           object
merchant                         object
category                       category
amt                             float64
first                            object
last                             object
gender                         category
street                           object
city                             object
state                          category
zip                              object
lat                             float64
long                            float64
city_pop                          int64
job                            category
dob                      datetime64[ns]
trans_num                        object
unix_time                         int64
merch_lat                       float64
merch_long                      float64
is_fraud                           int8
merch_zipcode           

## Step 3 — Further Memory Optimisation

The remaining high-memory columns are string object columns.
`merchant` and `city` have repeated values and benefit from category casting.
`first`, `last`, `street`, `trans_num` are high-cardinality identifiers — we drop
`first`, `last`, and `street` entirely as they carry no analytical value and are
personally identifiable information (PII) with zero business signal.
`unix_time` is redundant — we already have `trans_date_trans_time` as a parsed datetime.

In [5]:
# drop PII columns and redundant time column
# first, last, street are personal identifiers with no analytical signal
# unix_time is redundant given trans_date_trans_time is already parsed
df.drop(columns=["first", "last", "street", "unix_time"], inplace=True)

# merchant and city have repeated values — category dtype will compress them
df["merchant"] = df["merchant"].astype("category")
df["city"]     = df["city"].astype("category")

# confirm final memory
memory_mb = df.memory_usage(deep=True).sum() / 1e6
print(f"Memory after full optimisation : {memory_mb:.1f} MB")
print(f"Shape                          : {df.shape}")
print(f"Columns remaining              : {df.shape[1]}")
print(f"\nFinal columns:")
print(df.columns.tolist())

Memory after full optimisation : 426.4 MB
Shape                          : (1296675, 19)
Columns remaining              : 19

Final columns:
['trans_date_trans_time', 'cc_num', 'merchant', 'category', 'amt', 'gender', 'city', 'state', 'zip', 'lat', 'long', 'city_pop', 'job', 'dob', 'trans_num', 'merch_lat', 'merch_long', 'is_fraud', 'merch_zipcode']


## Step 4 — Null & Duplicate Audit

Before saving we need to confirm two things:
1. No unexpected nulls exist in columns we will use for analysis
2. No duplicate transactions exist in the dataset

`trans_num` is the unique transaction identifier — we use it to detect duplicates.
Any nulls in core analytical columns (amt, category, state, is_fraud) would require
a decision on how to handle them before downstream analysis.

In [6]:
# null audit across all columns
null_counts = df.isnull().sum()
null_pct    = (null_counts / len(df) * 100).round(2)

null_report = pd.DataFrame({
    "null_count"  : null_counts,
    "null_percent": null_pct
}).sort_values("null_count", ascending=False)

print("Null audit:")
print(null_report[null_report["null_count"] > 0].to_string()
      if null_report["null_count"].sum() > 0
      else "No nulls found in any column")

# duplicate audit using transaction id
duplicate_count = df.duplicated(subset=["trans_num"]).sum()
print(f"\nDuplicate transactions (by trans_num) : {duplicate_count:,}")

# duplicate audit on full row
full_duplicate_count = df.duplicated().sum()
print(f"Full row duplicates                   : {full_duplicate_count:,}")

Null audit:
No nulls found in any column

Duplicate transactions (by trans_num) : 0
Full row duplicates                   : 0


## Step 5 — Date Range & Fraud Rate Profile

Before saving, we capture three key facts about the dataset:
1. The transaction date range — confirms what time period we are analysing
2. Overall fraud rate — sets the baseline for all downstream fraud-related analysis
3. Transaction volume by year — confirms data is not heavily skewed to one period

These numbers will be referenced throughout all subsequent notebooks.

In [7]:
# date range
date_min = df["trans_date_trans_time"].min()
date_max = df["trans_date_trans_time"].max()
date_span = (date_max - date_min).days

print("Dataset profile:")
print(f"  Date range        : {date_min.date()} to {date_max.date()}")
print(f"  Span              : {date_span} days ({date_span/365:.1f} years)")
print(f"  Total transactions: {len(df):,}")
print(f"  Unique customers  : {df['cc_num'].nunique():,}")
print(f"  Unique merchants  : {df['merchant'].nunique():,}")
print(f"  Unique categories : {df['category'].nunique():,}")
print(f"  Unique states     : {df['state'].nunique():,}")

# fraud baseline
fraud_rate = df["is_fraud"].mean() * 100
fraud_count = df["is_fraud"].sum()
print(f"\nFraud baseline:")
print(f"  Fraudulent transactions : {fraud_count:,}")
print(f"  Fraud rate              : {fraud_rate:.2f}%")

# transaction volume by year
print("\nTransaction volume by year:")
print(df.groupby(df["trans_date_trans_time"].dt.year)["amt"]
        .agg(transactions="count", total_spend="sum")
        .assign(total_spend=lambda x: x["total_spend"].map("${:,.0f}".format))
        .to_string())

Dataset profile:
  Date range        : 2019-01-01 to 2020-06-21
  Span              : 537 days (1.5 years)
  Total transactions: 1,296,675
  Unique customers  : 983
  Unique merchants  : 693
  Unique categories : 14
  Unique states     : 51

Fraud baseline:
  Fraudulent transactions : 7,506
  Fraud rate              : 0.58%

Transaction volume by year:
                       transactions  total_spend
trans_date_trans_time                           
2019                         924850  $64,984,953
2020                         371825  $26,237,476


## Step 6 — Save to Parquet

Save the validated, optimised dataframe as Parquet.
All downstream notebooks (02 through 07) load from this file — never from the raw CSV.

Parquet advantages over CSV for this project:
- Preserves dtypes exactly — no re-casting needed in subsequent notebooks
- Significantly smaller file size
- Loads 10x faster than CSV at this row count

In [8]:
# save to parquet
df.to_parquet(PARQUET_PATH, index=False, engine="pyarrow")

# confirm file was written and report size
file_size_mb = os.path.getsize(PARQUET_PATH) / 1e6
print(f"Parquet saved successfully")
print(f"  Path      : {PARQUET_PATH}")
print(f"  File size : {file_size_mb:.1f} MB")
print(f"  Rows      : {len(df):,}")
print(f"  Columns   : {df.shape[1]}")

# verify it reads back correctly
df_verify = pd.read_parquet(PARQUET_PATH, engine="pyarrow")
print(f"\nRead-back verification:")
print(f"  Shape match : {df_verify.shape == df.shape}")
print(f"  Dtype match : {(df_verify.dtypes == df.dtypes).all()}")

Parquet saved successfully
  Path      : ../data/processed/transactions_clean.parquet
  File size : 95.3 MB
  Rows      : 1,296,675
  Columns   : 19

Read-back verification:
  Shape match : True
  Dtype match : True
